In [ ]:
print('===== EXPORT PREBUILT NATIVE .SO (build 1 lan, cac lan sau khong build) =====')
print('===== FIX LOSS TEST FAST (parallel gcc -O2, same numerics) =====')
print('===== FIX LOSS TEST: lr 0.05 -> 0.0006, 20 steps, expect loss < 10.5 =====')
# 1/4 - clone repos (fresh), dotnet
print('===== PARITY V3 (tied lm_head fixed, fresh file) =====')

# 1/4 - clone + dotnet (fault-tolerant)
# 1/4 - clone + dotnet (fault-tolerant)
print('===== ALL NATIVE KERNELS -> PURE TKV (14+7+2) + LOSS CHECK =====')
# 1/4 - clone main + prototype + Inference, install .NET (fault-tolerant)
import subprocess, os, shutil
print('===== INTEGRATE 14 PURE TKV KERNELS INTO NATIVE BUILD + LOSS CHECK =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
subprocess.run('rm -rf /content/TVinf', shell=True)
subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git /content/TVinf', shell=True, capture_output=True, text=True, timeout=600)
dotnet=shutil.which('dotnet')
if not dotnet:
    print('dotnet not found -> installing .NET 10 SDK...')
    p=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && chmod +x /tmp/dotnet-install.sh && /tmp/dotnet-install.sh --channel 10.0 --install-dir /usr/share/dotnet', shell=True, capture_output=True, text=True, timeout=900)
    print('install rc=',p.returncode)
    dotnet='/usr/share/dotnet/dotnet'
print('dotnet =',dotnet,'exists=',os.path.exists(dotnet))

In [ ]:
# 2/4 - build native all-TKV .so + run DEBUG bseq (dump logits stats)
import subprocess, shutil, os, re
root='/content/TVmain/Release/tools/simd_native'
ex='/content/TVcpu/Release/cuda_backend/examples'
bwd=['bias_add_rows_cpu.cpu.c','softmax_causal_fwd_pure.cpu.c','merge_hb_head_cpu.cpu.c',
     'layernorm_bwd_cpu.cpu.c','gelu_erf_bwd_pure.cpu.c','gelu_erf_fwd_pure.cpu.c',
     'bias_grad_cpu.cpu.c','softmax_causal_bwd_pure.cpu.c','embed_bwd_pure.cpu.c',
     'layernorm_fwd_cpu.cpu.c']
parts=[]
for f in bwd:
    s=open(ex+'/'+f).read()
    ls=[l for l in s.split('\n') if not l.startswith('#include') and not l.startswith('#define') and l.strip()!='']
    parts.append('\n'.join(ls))
hdr='#include <immintrin.h>\n#include <math.h>\n#include <stdlib.h>\n#define EXPORT __attribute__((visibility("default")))\n'
open(root+'/tkv_bwd_ops.c','w').write(hdr+'\n\n'.join(parts)+'\n')
shutil.copyfile(ex+'/linear_dw_cpu.cpu.c',root+'/tkv_linear_dw.c')
shutil.copyfile(ex+'/linear_dx_cpu.cpu.c',root+'/tkv_linear_dx.c')
p=root+'/tkv_simd_native.c'; s=open(p).read()
for name,gf in [('F32ListAdd','f32_list_add_cpu.cpu.c'),('F32ListCopy','f32_list_copy_cpu.cpu.c')]:
    s=re.sub(r'EXPORT int '+name+r'\(.*?\n\}\n',open(ex+'/'+gf).read().strip()+'\n',s,count=1,flags=re.S)
open(p,'w').write(s)
shutil.copyfile(ex+'/flash_all_cpu.cpu.c',root+'/tkv_flash_attn.c')
shutil.copyfile(ex+'/attn_cuda_stub_cpu.cpu.c',root+'/tkv_attn_cuda.c')
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
# BIEN DICH SONG SONG: moi file 1 gcc -O2, roi link. Giu nguyen -O2 -> khong doi numerics.
import time
t0=time.time()
procs=[]
for sf in srcs.split():
    of=sf[:-2]+'.fast.o'
    pr=subprocess.Popen('gcc -O2 -mavx2 -mfma -fPIC -c '+sf+' -o '+of,
                        shell=True,cwd=root,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    procs.append((sf,pr))
errs=[]
for sf,pr in procs:
    so,se=pr.communicate(timeout=600)
    if pr.returncode!=0: errs.append(sf+': '+se[:800])
print('compile parallel rc errors=',errs,' elapsed=%.1fs'%(time.time()-t0))
assert not errs, errs
objs=' '.join([sf[:-2]+'.fast.o' for sf in srcs.split()])
r=subprocess.run('cd '+root+' && gcc -shared -o tkv_simd_native.so '+objs+' -lm -lpthread -ldl',
                 shell=True,capture_output=True,text=True,timeout=600)
print('link rc=',r.returncode,' total build=%.1fs'%(time.time()-t0))
print('build rc=',r.returncode)
assert r.returncode==0, r.stderr[:2000]
print('READY .so bytes=',os.path.getsize(root+'/tkv_simd_native.so'))


In [ ]:
# 2/4 - export .so dang base64 (de assistant luu vao repo)
import base64, os
p='/content/TVmain/Release/tools/simd_native/tkv_simd_native.so'
b=open(p,'rb').read()
print('SO_SIZE',len(b))
s=base64.b64encode(b).decode('ascii')
CH=100000
print('SO_B64_BEGIN')
for i in range(0,len(s),CH):
    print('SO_B64_CHUNK:'+s[i:i+CH])
print('SO_B64_END')

In [ ]:
# 3/4 - summary
print('Gui notebook nay lai de assistant luu .so vao repo.')